In [ ]:
import os, time
import pandas as pd
from dotenv import load_dotenv
from sodapy import Socrata

load_dotenv()
client = Socrata("data.cityofnewyork.us", os.getenv("SOCRATA_TOKEN"), timeout=120)

os.makedirs("ems_raw", exist_ok=True)

cols = "incident_datetime,zipcode,initial_severity_level_code,valid_incident_rspns_time_indc"
PAGE = 10000   # smaller pages are far less likely to time out into a 503

def get_page(where, offset, retries=8):
    for i in range(retries):
        try:
            return client.get("76xm-jjuj", select=cols, where=where,
                              order="incident_datetime", limit=PAGE, offset=offset)
        except Exception as e:
            wait = min(2 ** i, 60)
            print(f"  offset {offset}: {e} -> retry {i+1}/{retries} in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"Gave up at offset {offset}")

for year in range(2016, 2023):
    final = f"ems_raw/{year}.csv"
    if os.path.exists(final):
        continue
    part_dir = f"ems_raw/{year}_parts"
    os.makedirs(part_dir, exist_ok=True)
    w = f"incident_datetime between '{year}-01-01T00:00:00' and '{year}-12-31T23:59:59'"

    offset = 0
    while True:
        part = f"{part_dir}/{offset:09d}.csv"
        if os.path.exists(part):                  # page already downloaded
            offset += PAGE
            continue
        batch = get_page(w, offset)
        if not batch:
            break
        pd.DataFrame.from_records(batch).to_csv(part, index=False)
        offset += PAGE
        print(f"{year}: {offset:,} rows requested")

    parts = sorted(os.listdir(part_dir))
    if parts:
        pd.concat((pd.read_csv(f"{part_dir}/{p}", dtype={"zipcode": str}) for p in parts),
                  ignore_index=True).to_csv(final, index=False)
        print(year, "saved")

2017: 10,000 rows requested
2017: 20,000 rows requested
2017: 30,000 rows requested
2017: 40,000 rows requested
2017: 50,000 rows requested
2017: 60,000 rows requested
2017: 70,000 rows requested
2017: 80,000 rows requested
2017: 90,000 rows requested
2017: 100,000 rows requested
2017: 110,000 rows requested
2017: 120,000 rows requested
2017: 130,000 rows requested
2017: 140,000 rows requested
2017: 150,000 rows requested
2017: 160,000 rows requested
2017: 170,000 rows requested
2017: 180,000 rows requested
2017: 190,000 rows requested
2017: 200,000 rows requested
2017: 210,000 rows requested
2017: 220,000 rows requested
2017: 230,000 rows requested
2017: 240,000 rows requested
2017: 250,000 rows requested
2017: 260,000 rows requested
2017: 270,000 rows requested
2017: 280,000 rows requested
2017: 290,000 rows requested
2017: 300,000 rows requested
2017: 310,000 rows requested
2017: 320,000 rows requested
2017: 330,000 rows requested
2017: 340,000 rows requested
2017: 350,000 rows requ

## Settings
Everything you might change is here. **The defaults fit the LinuxONE VM** (8.4 GB RAM, about 5.6 GB free, 3 CPUs): two years of data peak at about 1.6 GB for the panel stage (measured on the VM) and roughly 3.5 GB for the whole notebook (estimated). All seven years will not fit there and will kill the kernel.
On a machine with more memory, widen `EMS_YEARS` and move `TRAIN_START` earlier. If you change `EMS_YEARS`, change the three dates to match: the test period must be inside the loaded years, and training must have at least a few months.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
from grease.forecast import features as F, baseline as B, evaluate as E, zones as Z, export as X
from grease.forecast.lgbm import ForecastModel
from grease import risk

EMS_YEARS = [2021, 2022]                         # SAFE DEFAULT for the LinuxONE VM (8.4 GB RAM, ~5.6 GB free, 3 CPUs). On a bigger machine try list(range(2016, 2023))
TEST_START = pd.Timestamp("2022-01-01")          # F11: test on 2022
VAL_START = pd.Timestamp("2021-10-01")           # last quarter of 2021 = validation set (early stopping)
TRAIN_START = pd.Timestamp("2021-01-01")          # with more years loaded, move this earlier (e.g. 2016-01-01)
EXCLUDE_YEARS = []                               # F10: e.g. [2020] to drop COVID from training
FEATURE_GROUPS = F.DEFAULT_GROUPS                # F5-F7: also try adding "weather_extra", "zone_census", "zone_health_access"
PARAMS = {}                                      # LightGBM overrides, e.g. {"n_estimators": 300, "learning_rate": 0.1} for a quick run
DATA = Path("data")                              # notebooks/data

## Panel and features
One row per **ZIP code x 30-minute bucket**, built by streaming one year's CSV at a time (the ~10 million raw rows are never all in memory). A row is an *issue time* (the end of that bucket): `y`, `y_critical`,
the lags and rolling sums (which include the current bucket) and the time features are all known by then. Critical = initial severity code 1-3 (assumption F9, to be confirmed).
Logic lives in `src/grease/forecast/features.py`, which is unit-tested.

In [ ]:
panel, rows_read = F.build_panel_from_csvs([f"ems_raw/{y}.csv" for y in EMS_YEARS])

# completeness check against the API's yearly totals: a short year silently biases everything downstream
API_TOTALS = {2016: 1441211, 2017: 1463446, 2018: 1530922, 2019: 1533251, 2020: 1412701, 2021: 1491454, 2022: 1583531}
check = pd.DataFrame({"rows": {y: rows_read[f"ems_raw/{y}.csv"] for y in EMS_YEARS}, "api_total": {y: API_TOTALS[y] for y in EMS_YEARS}})
check["pct_complete"] = (100 * check.rows / check.api_total).round(1)
print(check)
short = check[check.pct_complete < 99].index.tolist()
if short:
    print(f"WARNING: {short} look incomplete. Delete ems_raw/<year>.csv and ems_raw/<year>_parts, then rerun the fetch cell.")

panel = F.add_history_features(panel)   # lags and rolling sums as of the issue bucket, plus time features
print(panel.shape, panel.bucket.min(), "->", panel.bucket.max(), "| zipcodes:", panel.zipcode.nunique(), f"| ~{panel.memory_usage().sum() / 1e9:.1f} GB")

## Link all other data sources onto it (`full`)
EMS counts are combined with everything fetched and cleaned by **`DarraghC.ipynb`** (run that notebook first; its outputs live in `notebooks/data/`):

| Joined on | From | Columns added |
|---|---|---|
| 30-minute `bucket` | `weather_calendar_30min.csv.gz` | weather and holiday columns, and the `eval_` heat / heavy-rain day flags |
| `zipcode` -> `zone` -> | `interim/zip_to_zone.csv` then `interim/zone_context.csv` | borough, location, population, asthma rate, Census columns, hospital access |

**Only the columns the chosen `FEATURE_GROUPS` need are attached** (plus the two `eval_` flags), and each is built from one small lookup repeated over the grid, so `full` stays lean. Add a group to `FEATURE_GROUPS` and rerun from here to test it.
`eval_*` columns only slice results by heat or heavy-rain days and are never model inputs. Which columns the model uses is an explicit list (decision M2), not "everything in the table".

In [ ]:
wx = pd.read_csv(DATA / "weather_calendar_30min.csv.gz", parse_dates=["bucket"]).set_index("bucket")
zip_to_zone = pd.read_csv(DATA / "interim" / "zip_to_zone.csv", dtype=str).set_index("zipcode")["zone"]
zone_ctx = pd.read_csv(DATA / "interim" / "zone_context.csv", dtype={"zone": str}).set_index("zone")

full = F.link_context(panel, wx, zip_to_zone, zone_ctx, FEATURE_GROUPS)
added = [c for c in full.columns if c not in panel.columns]
del panel
zone_known = full["borough"].notna().mean() if "borough" in full else float("nan")
print("full:", full.shape, f"| ~{full.memory_usage().sum() / 1e9:.1f} GB | added columns:", added)
print(f"rows whose ZIP is in a known zone: {zone_known:.1%} (the rest are rare ZIPs, about 0.1% of incidents)")
full.head(3).T

## Model: expected demand for the next 30 min / 1 h / 2 h / 3 h (decisions D8-D10)
For every ZIP and every issue time we predict the **expected number of incidents in the next window**, separately for **critical** and **other** calls (8 LightGBM Poisson models: 2 acuities x 4 horizons).
The risk index (further down) is a derived, explainable layer on top, not part of the model. Code: `src/grease/forecast/` and `src/grease/risk.py`.

- **Split by date** (F11): train up to the validation start, validate for early stopping, test on 2022. A gap of 3 hours is dropped before each boundary because a training target looks up to 3 hours ahead.
- **Baselines** (M4), scored on exactly the same rows: historical mean for that ZIP x weekday x time of day; that mean x a citywide recent-surge multiplier; and the same window one week earlier.
- **Memory:** targets are computed one at a time as arrays, the feature matrix is built once as float32, and all 8 models share one binned dataset.

In [ ]:
features = F.select_features(full.columns, FEATURE_GROUPS)
print(len(features), "features:", features)

b = full["bucket"]
embargo = pd.Timedelta(minutes=30 * max(F.HORIZONS.values()))     # keep a training target from reaching into the next period
complete = F.complete_window_mask(full)                           # the last 3 h of each ZIP have no full future window
in_train = ((b >= TRAIN_START) & (b < VAL_START - embargo)).to_numpy() & ~b.dt.year.isin(EXCLUDE_YEARS).to_numpy() & complete
in_valid = ((b >= VAL_START) & (b < TEST_START - embargo)).to_numpy() & complete
in_test = (b >= TEST_START).to_numpy()
rows_train, rows_valid, rows_test = (np.flatnonzero(m) for m in (in_train, in_valid, in_test))
for name, r in (("train", rows_train), ("validation", rows_valid), ("test", rows_test)):
    assert len(r), f"{name} period is empty: the loaded EMS_YEARS do not cover it. Adjust TRAIN_START / VAL_START / TEST_START in the settings cell."
    print(f"{name:10s} {len(r):>12,} rows   {b.iloc[r].min()} -> {b.iloc[r].max()}")

In [ ]:
model = ForecastModel(features, PARAMS).fit(full, rows_train, rows_valid, targets=lambda a, h: F.target_array(full, a, h))

In [ ]:
pred = model.predict(full, rows_test)       # pred_critical_30m ... pred_other_3h, pred_all_30m ... (expected counts, not integers)
test = full.iloc[rows_test]
train_small = full.iloc[rows_train][["zipcode", "dow", "hour", "bucket", "roll_2h", "y"]]   # only what the baselines need
surge = B.surge_multiplier(train_small, test)
results = {}                                # horizon -> actuals and every model's predictions, on the same rows
for h in F.HORIZONS:
    t_all = F.target_array(full, "all", h)
    hist = B.historical_mean(train_small, t_all[rows_train], test)
    results[h] = dict(actual=t_all[rows_test], preds={
        "model": pred[f"pred_all_{h}"].to_numpy(), "hist_mean": hist, "hist_x_surge": hist * surge,
        "last_week": B.last_week(full, t_all)[rows_test]})
print("predicted", f"{len(pred):,}", "test rows for horizons", list(F.HORIZONS))

In [ ]:
slices = {"all test rows": np.ones(len(test), bool),
          "heat days (daily max >= 32C)": (test.eval_is_heat_day == 1).to_numpy(),
          "heavy-rain days": (test.eval_is_heavy_rain_day == 1).to_numpy(),
          "busiest 20 zipcodes": test.zipcode.isin(E.busiest_zipcodes(train_small, 20)).to_numpy()}
if "borough" in test:
    for bname in sorted(test.borough.dropna().unique()):
        slices[f"borough: {bname}"] = (test.borough == bname).to_numpy()

scores = pd.concat([E.score_slices(r["actual"], r["preds"], slices).assign(horizon=h) for h, r in results.items()], ignore_index=True)
scores["horizon"] = pd.Categorical(scores["horizon"], categories=list(F.HORIZONS), ordered=True)

mae = scores.pivot_table(index=["slice", "horizon"], columns="model", values="MAE", observed=True)
mae["model beats best baseline by (%)"] = (100 * (1 - mae["model"] / mae[["hist_mean", "hist_x_surge", "last_week"]].min(axis=1))).round(1)
print("MAE of the expected count (lower is better). Positive % = the model has lower error than the best baseline.")
display(mae[["model", "hist_mean", "hist_x_surge", "last_week", "model beats best baseline by (%)"]].round(4))
print("Poisson deviance (lower is better):")
display(scores.pivot_table(index=["slice", "horizon"], columns="model", values="deviance", observed=True).round(4))
print("Bias of the model (mean predicted / mean actual; 1.0 = unbiased), per slice and horizon:")
display(scores[scores.model == "model"].pivot_table(index="slice", columns="horizon", values="bias", observed=True).round(3))

In [ ]:
for h in ("1h", "3h"):
    print(f"Calibration, next {h}, all calls: mean predicted vs mean actual by prediction decile (ratio should be close to 1 in every row)")
    display(E.calibration_table(results[h]["actual"], results[h]["preds"]["model"]).round(4))
imp = model.feature_importance()
print("Feature importance (share of total gain, averaged over the 8 models):")
display(imp.mean(axis=1).sort_values(ascending=False).head(15).round(3))

In [ ]:
t0 = test.groupby("bucket")["y"].sum().idxmax()                       # the busiest half-hour in the test period
rows = test.index[test.bucket == t0]
snap = pred.loc[rows].assign(zipcode=test.loc[rows, "zipcode"].to_numpy(), bucket=t0)
zones_now = Z.aggregate_to_zones(snap, zip_to_zone, ["pred_critical_1h", "pred_other_1h"]).set_index("zone")   # M7: ZIP -> zone, exact sum
zones_now["risk_index"] = risk.risk_index(zones_now.pred_critical_1h, zones_now.pred_other_1h, risk.vulnerability(zone_ctx)).round(1)
zones_now = zones_now.join(zone_ctx[["borough"]]).sort_values("risk_index", ascending=False)
print("Snapshot at", t0, "- expected incidents in the next hour per zone, and the 0-100 risk index (D9)")
zones_now.head(10).round(2)

## Hand-off table for the optimizer and simulator
One row per **zone x 30-minute bucket** of the test period: the model's expected incidents (`pred_*`), the scheduled-policy demand (`hist_all_*`), what actually happened (`actual_*`, for the oracle policy and scoring) and the risk index.
Columns and how to use them are described in `docs/forecast_handoff.md`. Saved next to the models (git-ignored, regenerate by rerunning this notebook).

In [ ]:
actual = {(a, h): F.target_array(full, a, h)[rows_test] for a in F.ACUITY for h in F.HORIZONS}
baseline = {h: results[h]["preds"]["hist_mean"] for h in F.HORIZONS}
zone_table = X.zone_forecast_table(test, pred, actual, baseline, zip_to_zone, zone_ctx)
out = DATA / "processed"
out.mkdir(parents=True, exist_ok=True)
zone_table.to_csv(out / "forecast_zone_test.csv.gz", index=False, float_format="%.3f", compression={"method": "gzip", "compresslevel": 1})
print("saved", out / "forecast_zone_test.csv.gz", "|", f"{len(zone_table):,} rows,", zone_table.zone.nunique(), "zones,", len(zone_table.columns), "columns")
zone_table.head(3).T

In [ ]:
out = DATA / "processed"
out.mkdir(parents=True, exist_ok=True)
model.save(out / "forecast_models")             # LightGBM text files + meta.json: portable to the VM
scores.to_csv(out / "forecast_scores.csv", index=False)
print("saved models and scores to", out)